In [22]:
import torch
import numpy as np
import re
import json
import emoji
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from collections import defaultdict
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from tqdm import tqdm

In [17]:
config = {
    'data_dir': './go_emotions',
    'model_name': 'roberta-base',
    
    'num_labels':28,
    'max_length': 128,
    'batch_size': 16,

    'focal_alpha': 1,
    'focal_gamma': 2,

    'num_train_epochs': 6,
    'learning_rate': 2e-5,
    'weight_decay': 0.01,
    'max_grad_norm': 1.0
}

# Test model

In [5]:
with open("emotion_config.json", "r") as f:
    config = json.load(f)

emotion_labels = config["emotion_labels"]
best_thresholds = np.array(config["best_thresholds"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = r"./results\checkpoint-10864"

In [6]:
def preprocess_text(text):
    text = emoji.demojize(text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'/?u/\S+', '[USER]', text)
    text = re.sub(r'/?r/\S+', '[SUBREDDIT]', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.to(device)
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50267, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [8]:
def predict_emotion(text, thresholds):
    clean_text = preprocess_text(text)
    inputs = tokenizer(
        clean_text, 
        return_tensors="pt", 
        truncation=True, 
        max_length=128, 
        padding="max_length"
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.sigmoid(logits).cpu().numpy()[0]
        
    predictions = (probs >= thresholds).astype(int)
    
    results = [
        {"emotion": emotion_labels[i], "score": float(probs[i])}
        for i, val in enumerate(predictions) if val == 1
    ]
    
    if not results:
        max_idx = np.argmax(probs)
        results.append({"emotion": emotion_labels[max_idx], "score": float(probs[max_idx]), "note": "Below threshold"})
        
    return results

In [9]:
# Test model
test_sentences = [
    "Thank you so much for your help! You are amazing.",
    "I am so sorry for what happened, I feel terrible.",
    "That is so funny! I can't stop laughing.",
    "What is going on here? I'm so confused."
]

print(f"{'Input Text':<50} | {'Predicted Emotions'}")
print("-" * 80)

for text in test_sentences:
    res = predict_emotion(text, best_thresholds)
    emotions_str = ", ".join([f"{r['emotion']} ({r['score']:.2f})" for r in res])
    print(f"{text[:48]+'...':<50} | {emotions_str}")

Input Text                                         | Predicted Emotions
--------------------------------------------------------------------------------
Thank you so much for your help! You are amazing... | admiration (0.61), gratitude (0.84)
I am so sorry for what happened, I feel terrible... | remorse (0.52), sadness (0.42)
That is so funny! I can't stop laughing....        | amusement (0.81)
What is going on here? I'm so confused....         | confusion (0.83)


In [10]:
while True:
    user_input = input("\nEnter your sentence ('q' to exit): ")
    if user_input.lower() == 'q':
        break
    
    prediction = predict_emotion(user_input, best_thresholds)
    print("Result:")
    for p in prediction:
        print(f" - {p['emotion']}: {p['score']:.2f}")


Enter your sentence ('q' to exit):  q


# Visualize

In [20]:
def extract_embeddings_and_labels(texts, labels_list, batch_size=16):
    """
    Hàm lấy CLS embeddings từ model RoBERTa.
    """
    embeddings = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Extracting Embeddings"):
            batch_texts = texts[i:i+batch_size]
            clean_texts = [preprocess_text(t) for t in batch_texts]
            
            inputs = tokenizer(
                clean_texts, 
                return_tensors="pt", 
                truncation=True, 
                max_length=128, 
                padding="max_length"
            ).to(device)
            
            # Yêu cầu model trả về hidden_states
            outputs = model(**inputs, output_hidden_states=True)
            
            # Lấy hidden state ở layer cuối cùng (chỉ số -1)
            last_hidden_state = outputs.hidden_states[-1]
            
            # Lấy vector của token [CLS] (token đầu tiên ở vị trí 0)
            cls_embeddings = last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.extend(cls_embeddings)
            
    return np.array(embeddings), np.array(labels_list)

def get_samples(df, emotion_labels_list, num_total_samples=2500):
    """
    Lấy mẫu từ DataFrame train_df đảm bảo xuất hiện đủ 28 nhãn.
    """
    selected_indices = []
    
    # Tạo mapping để biết mỗi nhãn có ở những dòng nào
    label_to_indices = defaultdict(list)
    for idx, row in df.iterrows():
        # Lấy nhãn đầu tiên trong list labels để phân loại
        first_label = row['labels'][0]
        label_to_indices[first_label].append(idx)
    
    # Bước 1: Đảm bảo lấy đủ 28 nhãn (mỗi nhãn ít nhất samples_per_label mẫu)
    num_labels = len(emotion_labels_list)
    samples_per_label = num_total_samples // num_labels
    
    for label_idx in range(num_labels):
        indices = label_to_indices.get(label_idx, [])
        if indices:
            # Lấy mẫu từ những dòng thuộc nhãn này
            n_take = min(len(indices), samples_per_label)
            selected_indices.extend(np.random.choice(indices, n_take, replace=False))
    
    # Bước 2: Bù cho đủ 2500 mẫu nếu Bước 1 chưa lấy đủ
    remaining_needed = num_total_samples - len(selected_indices)
    if remaining_needed > 0:
        remaining_pool = list(set(df.index) - set(selected_indices))
        additional_indices = np.random.choice(remaining_pool, remaining_needed, replace=False)
        selected_indices.extend(additional_indices)
        
    # Trích xuất dữ liệu cuối cùng
    sampled_df = df.loc[selected_indices].copy()
    
    # Chuyển ID nhãn số thành chữ để hiển thị trên plot sau này
    sampled_texts = sampled_df['text'].tolist()
    sampled_labels = [emotion_labels_list[l[0]] for l in sampled_df['labels']]
    
    return sampled_texts, sampled_labels

def preprocess_text(text):
    text = emoji.demojize(text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'/?u/\S+', '[USER]', text)
    text = re.sub(r'/?r/\S+', '[SUBREDDIT]', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [26]:
train = f"./{config['data_dir']}/train.tsv"

columns = ['text', 'labels', 'ID']
train_df = pd.read_csv(train, sep='\t', header=None, names=columns)

train_df['labels'] = train_df['labels'].apply(lambda x: [int(i) for i in str(x).split(',')])
train_df['text'] = train_df['text'].apply(preprocess_text)
train_texts, train_labels = get_samples(
    train_df, 
    emotion_labels, 
    num_total_samples=2500
)
# 2. Trích xuất Embeddings (Sử dụng hàm extract_embeddings_and_labels đã viết trước đó)
embeddings, labels = extract_embeddings_and_labels(train_texts, train_labels)
print(f"Đã chuẩn bị xong {len(embeddings)} vectors embedding.")

Extracting Embeddings: 100%|██████████| 157/157 [00:09<00:00, 15.75it/s]

Đã chuẩn bị xong 2500 vectors embedding.


In [27]:
def save_embeddings(embeddings, labels, texts, base_filename="goemotions_embedded"):
    """
    Lưu dữ liệu theo Phương án 1: Tách biệt vector và metadata
    """
    # 1. Lưu vector embedding (định dạng npy rất tối ưu cho mảng số)
    embedding_file = f"{base_filename}_embeddings.npy"
    np.save(embedding_file, embeddings)
    
    # 2. Lưu metadata (Text và Nhãn) để đối chiếu sau này
    metadata_file = f"{base_filename}_metadata.csv"
    metadata_df = pd.DataFrame({
        'text': texts,
        'label': labels
    })
    metadata_df.to_csv(metadata_file, index=False, encoding='utf-8')
    
    print(f"--- Đã lưu xong ---")
    print(f"1. File vector: {embedding_file} ({embeddings.shape})")
    print(f"2. File metadata: {metadata_file}")

# Thực thi lưu trữ
#save_embeddings(embeddings, labels, train_texts)

--- Đã lưu xong ---
1. File vector: goemotions_embedded_embeddings.npy ((2500, 768))
2. File metadata: goemotions_embedded_metadata.csv
